# BirdCLEF 2026 — Perch Submit v3
**Submission matches sample_submission.csv row_ids EXACTLY (source of truth: sprint4).**
Perch ONNX -> taxonomy -> BirdCLEF 234 species.


In [ ]:
# [1] Install onnxruntime
import subprocess, sys, os, glob
def _fd(p):
    d=f"/kaggle/input/{p}"
    if os.path.isdir(d): return d
    for r,d,f in os.walk("/kaggle/input"):
        if p in r: return r
    return d
WD=_fd("birdclef-perch-models")
try:
    import onnxruntime
except ImportError:
    wh=sorted(glob.glob(os.path.join(WD,"*.whl")))
    if wh:
        subprocess.check_call([sys.executable,"-m","pip","install","--no-deps",wh[0]],
            stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
        import onnxruntime
print(f"onnxruntime {onnxruntime.__version__} OK")


In [ ]:
# [2] Imports
import os as _os
_os.environ["KAGGLE_NO_INTERNET"]="1"
import numpy as np, pandas as pd, onnxruntime as ort, librosa, gc, time, re
from pathlib import Path


In [ ]:
# [3] Paths + Config
def fd(p):
    d=f"/kaggle/input/{p}"
    if _os.path.isdir(d): return d
    for r,_,_ in _os.walk("/kaggle/input"):
        if p in r: return r
    return d

COMP_DIR=fd("birdclef-2026")
MODEL_DIR=fd("birdclef-perch-models")
print(f"COMP: {COMP_DIR}")
print(f"MODEL: {MODEL_DIR}")

TEST_DIR       =f"{COMP_DIR}/test_soundscapes"
OUTPUT_PATH    ="/kaggle/working/submission.csv"
TAXONOMY_PATH  =f"{COMP_DIR}/taxonomy.csv"
SAMPLE_SUB_PATH=f"{COMP_DIR}/sample_submission.csv"
PERCH_ONNX     =f"{MODEL_DIR}/perch_v2.onnx"
PERCH_LABELS   =f"{MODEL_DIR}/labels.csv"

SR,DURATION=32000,5
SEGMENT_SAMPLES=SR*DURATION
BATCH_SIZE=32
AUDIO_EXTS={".ogg",".mp3",".wav",".flac",".m4a",".opus",".webm"}

sopts=ort.SessionOptions()
sopts.graph_optimization_level=ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sopts.intra_op_num_threads=8


In [ ]:
# [4] Load: Perch ONNX + taxonomy + sample_submission
perch_sess=ort.InferenceSession(PERCH_ONNX,sopts,providers=["CPUExecutionProvider"])
perch_in=perch_sess.get_inputs()[0].name
perch_out=[o.name for o in perch_sess.get_outputs()]
print(f"Perch outputs: {perch_out}")

perch_labels_df=pd.read_csv(PERCH_LABELS)
taxonomy_df=pd.read_csv(TAXONOMY_PATH)
sample_sub=pd.read_csv(SAMPLE_SUB_PATH)

SPECIES_COLS=[c for c in sample_sub.columns if c!="row_id"]
N_SPECIES=len(SPECIES_COLS)
species_to_idx={sp:i for i,sp in enumerate(SPECIES_COLS)}

print(f"Perch: {len(perch_labels_df)} | Taxonomy: {len(taxonomy_df)} | Species: {N_SPECIES} | Sample rows: {len(sample_sub)}")

# Parse row_ids: "BC2026_Test_0001_S05_20250227_010002_5" -> (soundscape, end_time)
def parse_row_id(rid):
    parts=rid.rsplit("_",1)
    if len(parts)==2 and parts[1].isdigit():
        return parts[0],int(parts[1])
    return rid,0

# Build unique soundscapes from expected row_ids
soundscape_times={}
for rid in sample_sub["row_id"]:
    sc,t=parse_row_id(rid)
    soundscape_times.setdefault(sc,[]).append(t)
UNIQUE_SOUNDSCAPES=list(soundscape_times.keys())
print(f"Unique soundscapes in sample_sub: {len(UNIQUE_SOUNDSCAPES)}")
if UNIQUE_SOUNDSCAPES:
    print(f"  First: {UNIQUE_SOUNDSCAPES[0]} -> times {sorted(soundscape_times[UNIQUE_SOUNDSCAPES[0]])[:5]}")


In [ ]:
# [5] Build Perch -> BirdCLEF mapping
tax_id_to_sci={}
tax_id_to_common={}
for _,row in taxonomy_df.iterrows():
    pid=str(row["primary_label"])
    tax_id_to_sci[pid]=str(row.get("scientific_name","")).lower().strip()
    tax_id_to_common[pid]=str(row.get("common_name","")).lower().strip()

perch_labels_list=[]
for i,row in perch_labels_df.iterrows():
    lbl=str(row.get("label",row.get("scientific_name",""))).lower().strip()
    perch_labels_list.append(lbl)

bc_to_perch={}
perch_to_bc={}
for bc_sp in SPECIES_COLS:
    bc_idx=species_to_idx[bc_sp]
    sci=tax_id_to_sci.get(bc_sp,bc_sp.lower().replace("_"," ")).strip()
    common=tax_id_to_common.get(bc_sp,"").strip()
    matched=False
    # 1) exact science name
    for pi,pl in enumerate(perch_labels_list):
        if pl==sci:
            bc_to_perch[bc_idx]=pi;perch_to_bc.setdefault(pi,[]).append(bc_idx);matched=True;break
    # 2) genus
    if not matched and " " in sci:
        genus=sci.split()[0]
        for pi,pl in enumerate(perch_labels_list):
            if pl.startswith(genus+" "):
                bc_to_perch[bc_idx]=pi;perch_to_bc.setdefault(pi,[]).append(bc_idx);matched=True;break
    # 3) exact common
    if not matched and common:
        for pi,pl in enumerate(perch_labels_list):
            if pl==common:
                bc_to_perch[bc_idx]=pi;perch_to_bc.setdefault(pi,[]).append(bc_idx);matched=True;break
    # 4) substring
    if not matched and common:
        for pi,pl in enumerate(perch_labels_list):
            if common in pl or pl in common:
                bc_to_perch[bc_idx]=pi;perch_to_bc.setdefault(pi,[]).append(bc_idx);matched=True;break

matched=len(bc_to_perch)
print(f"Matched: {matched}/{N_SPECIES}")
if matched<N_SPECIES:
    unm=[sp for sp in SPECIES_COLS if species_to_idx[sp] not in bc_to_perch]
    print(f"Unmatched ({len(unm)}): {unm[:10]}")


In [ ]:
# [6] Inference functions
def predict_file(file_path, needed_segments):
    """Run Perch on an audio file, return dict of end_time -> prob_vector."""
    y,sr=librosa.load(file_path,sr=None,mono=True)
    if sr!=SR: y=librosa.resample(y,orig_sr=sr,target_sr=SR)
    if len(y)<SEGMENT_SAMPLES: y=np.pad(y,(0,SEGMENT_SAMPLES-len(y)))

    # Extract non-overlapping 5s segments
    segs=[]
    for s in range(0,len(y)-SEGMENT_SAMPLES+1,SEGMENT_SAMPLES):
        segs.append(y[s:s+SEGMENT_SAMPLES])
    if not segs: segs.append(y[:SEGMENT_SAMPLES])
    segs=np.stack(segs).astype(np.float32)  # (N,160000)

    # Batch Perch inference
    all_bc=np.zeros((len(segs),N_SPECIES),dtype=np.float32)
    for i in range(0,len(segs),BATCH_SIZE):
        batch=segs[i:i+BATCH_SIZE]
        outs=perch_sess.run(perch_out,{perch_in:batch})
        od=dict(zip(perch_out,outs))
        logits=None
        for k in ["label","logits"]:
            if k in od: logits=od[k]; break
        if logits is None: continue
        p=1.0/(1.0+np.exp(-logits))
        bc_p=np.zeros((len(batch),N_SPECIES),dtype=np.float32)
        for bc_idx,pi in bc_to_perch.items():
            if pi<p.shape[1]: bc_p[:,bc_idx]=p[:,pi]
        all_bc[i:i+len(batch)]=bc_p

    # Map segment index -> end_time
    result={}
    for seg_i in range(len(segs)):
        end_t=(seg_i+1)*DURATION
        if end_t in needed_segments or not needed_segments:
            result[end_t]=all_bc[seg_i]
    return result


In [ ]:
# [7] Run inference per soundscape from sample_submission
t0=time.time()

# Find audio files in test dir
audio_files={}
for f in sorted(_os.listdir(TEST_DIR)):
    fpath=_os.path.join(TEST_DIR,f)
    if _os.path.isfile(fpath) and not f.startswith("."):
        ext=_os.path.splitext(f)[1].lower()
        if ext in AUDIO_EXTS:
            sid=_os.path.splitext(f)[0]
            audio_files[sid]=fpath

print(f"Audio files found: {len(audio_files)}")
if not audio_files:
    print("NOTE: Real test audio injected during Kaggle scoring only.")

# Predict for each unique soundscape in sample_submission
sc_predictions={}  # sc_name -> {end_time -> prob_vector}
for sc in UNIQUE_SOUNDSCAPES:
    if sc in audio_files:
        needed=set(soundscape_times.get(sc,[]))
        try:
            preds=predict_file(audio_files[sc],needed)
            sc_predictions[sc]=preds
            print(f"  [OK] {sc}: {len(preds)} segments")
        except Exception as e:
            print(f"  [ERR] {sc}: {e}")
            sc_predictions[sc]={}
    else:
        # No audio file — will be filled with zeros later
        sc_predictions[sc]={}
        if len(audio_files)==0:
            pass  # normal in dry-run
        else:
            print(f"  [MISS] {sc}: no audio file found")

elapsed=time.time()-t0
print(f"Inference: {elapsed:.0f}s")


In [ ]:
# [8] Build submission CSV — EXACT row_ids from sample_submission (SAME ORDER)
results=[]
for rid in sample_sub["row_id"]:
    sc,end_t=parse_row_id(rid)
    preds=sc_predictions.get(sc,{})
    if end_t in preds:
        probs=preds[end_t]
    else:
        probs=np.zeros(N_SPECIES,dtype=np.float32)
    results.append([rid]+probs.tolist())

sub=pd.DataFrame(results,columns=["row_id"]+SPECIES_COLS)
# Ensure species columns are float32
for col in SPECIES_COLS:
    sub[col]=pd.to_numeric(sub[col],errors="coerce").fillna(0.0).astype(np.float32)

# CRITICAL: verify match with sample_submission
assert sub.shape==sample_sub.shape, f"Shape: {sub.shape} vs {sample_sub.shape}"
assert list(sub.columns)==list(sample_sub.columns), "Column mismatch"
assert sub["row_id"].equals(sample_sub["row_id"]), "Row IDs mismatch!"

sub.to_csv(OUTPUT_PATH,index=False)

nums=sub[SPECIES_COLS]
active=int((nums.max(axis=0)>0.01).sum()) if len(sub)>0 else 0

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(sub)} | Cols: {len(sub.columns)}")
print(f"Row IDs match sample_sub: {sub['row_id'].equals(sample_sub['row_id'])}")
if len(sub)>0:
    print(f"Mean prob: {nums.values.mean():.6f}")
    print(f"Max prob:  {nums.values.max():.6f}")
    print(f"Active (>0.01): {active}/{N_SPECIES}")
else:
    print("Empty submission")


In [ ]:
# [9] Done
print("\n"+"="*50)
print("SUBMISSION READY")
print("="*50)
print(f"Species matched: {matched}/{N_SPECIES}")
print(f"Row IDs match sample_sub: {sub['row_id'].equals(sample_sub['row_id'])}")
print(f"Soundscapes with audio: {len(audio_files)}")
print(f"Total runtime: {time.time()-t0:.0f}s")
print("\nSubmit to competition!")
